In [1]:
%load_ext autoreload
%autoreload 2

# fast

> An optional compiled kernel for the pattern scan: the same totals as the numpy path, in one pass.

In [2]:
#| default_exp fast

In [3]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [4]:
#| export
import os
import numpy as np
from randseq.coding import pattern_is_self_rc

try:
    if os.environ.get("RANDSEQ_NO_NUMBA"):
        raise ImportError("disabled by RANDSEQ_NO_NUMBA")
    import numba
    from numba import njit, prange
    HAVE_NUMBA = True
except ImportError:
    HAVE_NUMBA = False

## Why a second implementation of the scan

`get_pattern_scores` in `00_core` scores one pattern in three numpy steps — window codes for every
position of both strands, a sort per row to collapse repeats, and `bincount`s over what is left.
Each step is a full pass over an intermediate array of `n_sequences x 2 x insert length`, and on a
30k-sequence library that is about 0.2 s per pattern, spread evenly over the steps. No single step
is slow, so no numpy rewrite of one step buys much.

The kernel here fuses all of it into one compiled pass over the sequences. For each window it
computes the code directly from the 2-bit bases, folds a self-reverse-complementary code onto
`min(code, rc)` with the digit formula, and skips the window if this sequence has already counted
that code — a *stamp* array recording the last sequence seen per code, which replaces the sort.
What it accumulates is exactly what `_motif_totals(*motif_occurrences(...))` returns, so
`get_pattern_scores` can take either path and nothing downstream can tell which ran.

Measured on the 30,108-sequence panel library over the 43 shipped patterns: numpy 8.9 s, this
kernel 2.4 s on one core and 0.7 s on eight. The first call in a process also pays a one-off
compile of about 4 s, cached on disk afterwards where the package directory is writable.

### It is optional

numba is not a hard dependency. Install it with the `fast` extra:

```sh
pip install "randseq[fast] @ git+https://github.com/dbikard/randseq.git"
```

Without it, `HAVE_NUMBA` is `False` and the scan runs the numpy path, which is the readable
statement of the algorithm and stays the reference this kernel is tested against. Setting the
environment variable `RANDSEQ_NO_NUMBA=1` before importing `randseq` forces the numpy path even
when numba is installed, which is how the two are compared.

In [5]:
#| export
if HAVE_NUMBA:
    @njit(inline="always", cache=True)
    def _rc_code(code, k):
        """Code of the reverse complement: the k base-4 digits read backwards, each 3 - d."""
        r = 0
        for _ in range(k):
            r = (r << 2) | (3 - (code & 3))
            code >>= 2
        return r

    @njit(inline="always", cache=True)
    def _scan_rows(fwd, rc, lo, hi, d1, sp, d2, fc, thr, fold, n, ndep, fsum, first, stamp):
        """Accumulate the totals of sequences lo..hi-1 into n, ndep, fsum and first."""
        k = d1 + d2
        w = d1 + sp + d2
        L = fwd.shape[1]
        for i in range(lo, hi):
            dep = 1 if fc[i] < thr else 0
            for strand in range(2):
                arr = fwd if strand == 0 else rc
                for s in range(L - w + 1):
                    code = 0
                    ok = True
                    for j in range(d1):
                        b = arr[i, s + j]
                        if b > 3:               # non-ACGT or padding: the window is rejected
                            ok = False
                            break
                        code = code * 4 + b
                    if ok:
                        for j in range(d1, d1 + sp):   # the spacer is undefined but must
                            if arr[i, s + j] > 3:       # still be ACGT, as in the numpy path
                                ok = False
                                break
                    if ok:
                        for j in range(d2):
                            b = arr[i, s + d1 + sp + j]
                            if b > 3:
                                ok = False
                                break
                            code = code * 4 + b
                    if not ok:
                        continue
                    if fold:
                        r = _rc_code(code, k)
                        if r < code:
                            code = r
                    if stamp[code] != i:        # first time this sequence carries this motif
                        stamp[code] = i
                        if n[code] == 0:
                            first[code] = i
                        n[code] += 1
                        ndep[code] += dep
                        fsum[code] += fc[i]

    @njit(cache=True)
    def _scan_serial(fwd, rc, d1, sp, d2, fc, thr, fold):
        nc = 4 ** (d1 + d2)
        n = np.zeros(nc, np.int64)
        ndep = np.zeros(nc, np.int64)
        fsum = np.zeros(nc, np.float64)
        first = np.full(nc, fwd.shape[0], np.int64)
        stamp = np.full(nc, -1, np.int64)
        _scan_rows(fwd, rc, 0, fwd.shape[0], d1, sp, d2, fc, thr, fold, n, ndep, fsum, first, stamp)
        return n, ndep, fsum, first

    @njit(parallel=True, cache=True)
    def _scan_parallel(fwd, rc, d1, sp, d2, fc, thr, fold, nchunks):
        # Each chunk of sequences gets its own counters, so threads never write to the same
        # array; the chunks are summed afterwards. A stamp is per chunk too, which is correct
        # because a sequence lives in exactly one chunk.
        nc = 4 ** (d1 + d2)
        nseq = fwd.shape[0]
        N = np.zeros((nchunks, nc), np.int64)
        D = np.zeros((nchunks, nc), np.int64)
        F = np.zeros((nchunks, nc), np.float64)
        M = np.full((nchunks, nc), nseq, np.int64)
        for c in prange(nchunks):
            stamp = np.full(nc, -1, np.int64)
            lo = c * nseq // nchunks
            hi = (c + 1) * nseq // nchunks
            _scan_rows(fwd, rc, lo, hi, d1, sp, d2, fc, thr, fold, N[c], D[c], F[c], M[c], stamp)
        first = np.full(nc, nseq, np.int64)
        for c in range(nchunks):
            for x in range(nc):
                if M[c, x] < first[x]:
                    first[x] = M[c, x]
        return N.sum(axis=0), D.sum(axis=0), F.sum(axis=0), first

In [6]:
#| export
def pattern_totals(encoded_library, pattern, fc_values, log2FC_thr=-1, parallel=True):
    """Per motif code: how many sequences carry it, how many of those are depleted, the sum of
    their log2FC, and the first sequence that carries it.

    The same four arrays as `_motif_totals(*motif_occurrences(...), ...)`, with the same types,
    computed in one compiled pass. `parallel` spreads the sequences over numba's threads; pass
    False inside a worker process that is itself one of several, so the machine is not asked
    for workers x threads at once.

    Raises if numba is not available -- callers check `HAVE_NUMBA` first.
    """
    if not HAVE_NUMBA:
        raise RuntimeError("numba is not available; install randseq[fast] or use the numpy path")
    d1, sp, d2 = pattern
    if 4 ** (d1 + d2) > np.iinfo(np.int32).max:
        raise ValueError(
            f"pattern {pattern} has {d1 + d2} defined bases; the motif code would overflow "
            f"int32 (max 15). Such a pattern is not usable anyway: it would need "
            f"{4 ** (d1 + d2):.3g} counting bins.")
    fwd, rc = (np.ascontiguousarray(a, dtype=np.uint8) for a in encoded_library)
    fc = np.ascontiguousarray(fc_values, dtype=np.float64)
    fold = pattern_is_self_rc(pattern)
    threads = numba.get_num_threads()
    # Every chunk holds four counter arrays of 4**(d1+d2) entries, so the chunk count is capped
    # by memory as well as by threads: at nine defined bases 32 chunks would be ~270 MB.
    nchunks = min(fwd.shape[0], 4 * threads, max(threads, (1 << 23) // 4 ** (d1 + d2)))
    if parallel and threads > 1 and nchunks > 1:
        n, ndep, fsum, first = _scan_parallel(fwd, rc, d1, sp, d2, fc, float(log2FC_thr), fold,
                                              nchunks)
    else:
        n, ndep, fsum, first = _scan_serial(fwd, rc, d1, sp, d2, fc, float(log2FC_thr), fold)
    return n, ndep.astype(np.float64), fsum, first

### What guarantees this is the same scan?

The kernel re-implements every rule the numpy path applies — rejecting a window with a non-ACGT
base or padding, folding the strands of a self-rc pattern, counting a motif once per sequence —
so each is checked here against `motif_occurrences` + `_motif_totals`: on the real library for
contiguous, balanced and asymmetric patterns, and on random ragged libraries containing `N`.
`tests/test_fast_scan.py` repeats the comparison over all 43 shipped patterns, and runs the whole
`get_pattern_scores` both ways.

In [7]:
#| hide
if HAVE_NUMBA:
    import os, random
    import pandas as pd
    from randseq.coding import encode_library
    from randseq.core import motif_occurrences, _motif_totals
    from randseq.utils import calculate_log2fc, get_lib_seq_context
    from randseq.example_data import get_example_data_dir

    def _numpy_totals(enc, fc, pat, thr=-1.0):
        si, mc = motif_occurrences(enc, pat)
        return _motif_totals(si, mc, np.asarray(fc, float), thr, 4 ** (pat[0] + pat[2]))

    def _same(a, b):
        test_eq(a[0].tolist(), b[0].tolist())                        # support
        test_eq(a[1].tolist(), b[1].tolist())                        # depleted
        assert np.allclose(a[2], b[2], rtol=0, atol=1e-9)            # log2FC sum
        seen = a[0] > 0
        test_eq(a[3][seen].tolist(), b[3][seen].tolist())            # first sequence

    _c = pd.read_csv(os.path.join(get_example_data_dir(), "countsTable.csv.gz"), index_col=0)
    _l2 = calculate_log2fc(_c[["MFDpir", "JJ1886_T0"]], reference_column="MFDpir")["JJ1886_T0"]
    _enc = encode_library(get_lib_seq_context(_l2.index, "GTCCTAGGTATAATACTAGT",
                                              "GTTTTAGAGCTAGAAATAGC"))
    for _pat in [(6, 0, 0), (5, 0, 0), (4, 4, 4), (3, 4, 3), (4, 4, 3), (2, 1, 3), (4, 8, 4)]:
        for _par in (True, False):
            _same(pattern_totals(_enc, _pat, _l2.values, -1, parallel=_par),
                  _numpy_totals(_enc, _l2.values, _pat))

    _rng = random.Random(0)
    for _ in range(30):            # ragged, some with N -- including N inside the spacer
        _seqs = ["".join(_rng.choice("ACGT" if _rng.random() < 0.7 else "ACGTN")
                         for _ in range(_rng.randint(8, 40))) for _ in range(_rng.randint(1, 30))]
        _fc = [_rng.uniform(-6, 2) for _ in _seqs]
        _pat = _rng.choice([(6, 0, 0), (3, 4, 4), (2, 1, 3), (1, 2, 2), (3, 2, 3), (4, 4, 3)])
        _same(pattern_totals(encode_library(_seqs), _pat, _fc, -1),
              _numpy_totals(encode_library(_seqs), _fc, _pat))
    print("kernel == numpy path: real library, 7 patterns x serial/parallel, and 30 ragged fuzz libraries")
    test_fail(lambda: pattern_totals(_enc, (16, 0, 0), _l2.values), contains="overflow")
else:
    print("numba not installed: nothing to compare")

kernel == numpy path: real library, 7 patterns x serial/parallel, and 30 ragged fuzz libraries
